In [2]:
import boto3
import pandas as pd
import numpy as np
import io
import re

# ─── S3 CONFIG ─────────────────────────────────────────────────────
BUCKET  = 'sagemakerstack-transactionsrawdatabucket6643d2da-q7nfqsc3jpnn'
S3_KEY  = 'merged/merged_raw.csv'
CHUNK_SIZE = 100_000  # safe preview size

# ─── LOAD SAMPLE FROM S3 ────────────────────────────────────────────
print("📦 Loading sample from S3...")
s3 = boto3.client('s3')
response = s3.get_object(Bucket=BUCKET, Key=S3_KEY)
body = response['Body']
df = pd.read_csv(io.TextIOWrapper(body, encoding='utf-8'), nrows=CHUNK_SIZE, low_memory=False)
print(f"✅ Sample loaded: {df.shape}")

# ─── CLEAN COLUMN NAMES ─────────────────────────────────────────────
df.columns = df.columns.str.strip()

# ─── FEATURE: UNIT VALUE (Local Amount / Net Weight) ────────────────
df['Unit Value'] = (
    pd.to_numeric(df['Local Amount'], errors='coerce') /
    pd.to_numeric(df['Net Weight'], errors='coerce')
).replace([np.inf, -np.inf], np.nan)

# ─── FEATURE: TOTAL DECLARED DUTY (%) ───────────────────────────────
df['Declared Duty %'] = (
    (pd.to_numeric(df['Customs Duty BHD'], errors='coerce') +
     pd.to_numeric(df['Excise Duty BHD'], errors='coerce') +
     pd.to_numeric(df['VAT BHD'], errors='coerce')) /
    pd.to_numeric(df['Local Amount'], errors='coerce')
).replace([np.inf, -np.inf], np.nan)

# ─── FEATURE: DATE PARTS ────────────────────────────────────────────
df['Registration Date'] = pd.to_datetime(df['Registration Date'], errors='coerce')
df['Reg Year']  = df['Registration Date'].dt.year
df['Reg Month'] = df['Registration Date'].dt.month
df['Reg Day']   = df['Registration Date'].dt.day
df['Reg Weekday'] = df['Registration Date'].dt.dayofweek

# ─── FEATURE: Language check on Commercial Description ─────────────
def is_arabic(text):
    return bool(re.search(r'[\u0600-\u06FF]', str(text)))

df['Is Arabic Description'] = df['Commercial Description'].apply(is_arabic)

# ─── FEATURE: Description Length (as proxy for detail) ──────────────
df['Description Length'] = df['Commercial Description'].astype(str).apply(len)

# ─── MISSING VALUES REPORT ──────────────────────────────────────────
missing_report = df.isnull().mean().sort_values(ascending=False)
print("\n🧼 Missing values (%):")
display(missing_report[missing_report > 0] * 100)

# ─── PREVIEW OF ENGINEERED DATA ─────────────────────────────────────
print("\n🔍 Sample of engineered features:")
display(df[['HSCode', 'Local Amount', 'Net Weight', 'Unit Value',
            'Declared Duty %', 'Is Arabic Description', 'Description Length']].head())

# ─── SAVE CLEAN SAMPLE (optional) ───────────────────────────────────
# df.to_csv("clean_sample.csv", index=False)


📦 Loading sample from S3...
✅ Sample loaded: (100000, 73)

🧼 Missing values (%):


Agent Name                100.000
Cashier Name              100.000
Declared Duty %            99.887
Excise Duty BHD            99.643
Excise Duty Rate           99.377
Specification Code         99.276
Pref                       96.400
Warehouse Code             96.188
Exporter Name              82.330
Exporter CR                82.330
Cashier ID                 70.171
Sup Unit                   53.781
Sup Amt                    53.781
Assigned Examiner          52.001
Invoice Amount             42.142
Receipt Time               39.271
Local Amount               27.532
Unit Value                 27.532
Customs Duty BHD           24.676
Consignee Name             17.670
Consignee CR               17.670
VAT BHD                    17.264
Gross Weight               15.817
VAT Rate                   15.540
Customs Duty Rate           9.560
Exit Officer Name           6.170
Exit Officer ID             6.075
Office Exit                 5.793
Exit Office Code            5.793
Exit Time     


🔍 Sample of engineered features:


,HSCode,Local Amount,Net Weight,Unit Value,Declared Duty %,Is Arabic Description,Description Length
0,76151090,NaN,500.0,NaN,NaN,False,12
1,44209090,NaN,1440.0,NaN,NaN,False,42
2,73089090,NaN,800.0,NaN,NaN,False,11
3,87042110,NaN,1928.0,NaN,NaN,False,28
4,87032432,NaN,2156.0,NaN,NaN,False,27


In [9]:
script_code = """
import boto3
import pandas as pd
import io
import os

# ─── CONFIG ─────────────────────────────────────────────
BUCKET      = 'sagemakerstack-transactionsrawdatabucket6643d2da-q7nfqsc3jpnn'
PREFIX      = '.csv/'
S3_KEY      = 'merged/cleaned_merged_raw.csv'
OUTPUT_FILE = 'cleaned_merged_raw.csv'  # Local file in notebook
EXPECTED_COLUMNS = 73
CHUNK_SIZE = 100_000

# ─── CLEAN START ────────────────────────────────────────
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

# ─── Column definitions ────────────────────────────────
string_cols = [
    'INDEX','Customs Office Code','Customs Office Name','Regime','Registration Serial',
    'Registration Number','Reference Number','Registration Date','Registration Time',
    'Declarant CR','Declarant Name','Agent ID','Agent Name','Consignee CR','Consignee Name',
    'Exporter CR','Exporter Name','Receipt Serial','Receipt Date','Receipt Time','Cashier ID',
    'Cashier Name','LOC Code','LOC Name','Terms of delivery','Office Exit','Office Exit Name',
    'Item Number','Procedure','CP3','Pref','HSCode','Commercial Description',
    'Country of Origin Code','Country of Origin','Country of Export Code','Country of Export',
    'Country of Destination Code','Country of Destination','Invoice Currency','Local Currency',
    'Currency','Package Code','Sup Unit','Exit Serial','Exit Number','Exit Date','Exit Time',
    'Exit Office Code','Exit Officer ID','Exit Officer Name','Status','Assigned Examiner',
    'First Reroute By','Specification Code','Warehouse Code'
]
dtype_map = { col: "string" for col in string_cols }

numeric_cols = [
    'Year','Invoice Amount','Local Amount','Gross Weight','Net Weight','Customs Duty Rate',
    'Customs Duty BHD','Excise Duty Rate','Excise Duty BHD','VAT Rate','VAT BHD','Total Duty BHD',
    'HS-Rate','Fees','Package Amt','Sup Amt'
]

# ─── Stream from S3 and merge ───────────────────────────
s3 = boto3.client('s3')
paginator = s3.get_paginator('list_objects_v2').paginate(Bucket=BUCKET, Prefix=PREFIX)

first_chunk = True
total_written = 0

for page in paginator:
    for obj in page.get('Contents', []):
        key = obj['Key']
        if not key.lower().endswith('.csv'):
            continue
        print(f"⏳ Processing {key}")
        body = s3.get_object(Bucket=BUCKET, Key=key)['Body']

        for chunk in pd.read_csv(io.TextIOWrapper(body, encoding='utf-8'),
                                 chunksize=CHUNK_SIZE,
                                 dtype=dtype_map,
                                 low_memory=False,
                                 on_bad_lines='skip'):
            # Drop rows with invalid column counts
            valid_chunk = chunk.loc[chunk.apply(lambda row: len(row) == EXPECTED_COLUMNS, axis=1)]

            # Coerce numerics
            for col in numeric_cols:
                if col in valid_chunk.columns:
                    valid_chunk[col] = pd.to_numeric(valid_chunk[col], errors='coerce')

            # Append to output
            valid_chunk.to_csv(
                OUTPUT_FILE,
                mode='a',
                header=first_chunk,
                index=False
            )
            total_written += len(valid_chunk)
            first_chunk = False

print(f"✅ Merging complete. {total_written:,} valid rows written.")
"""

with open("merge_cleaned.py", "w") as f:
    f.write(script_code.strip())

print("✅ Script saved as merge_cleaned.py")


✅ Script saved as merge_cleaned.py


In [10]:
!python merge_cleaned.py

⏳ Processing .csv/2019/2019_01_01 to 2019_01_31.csv
⏳ Processing .csv/2019/2019_02_01 to 2019_02_28.csv
⏳ Processing .csv/2019/2019_03_01 to 2019_03_31.csv
⏳ Processing .csv/2019/2019_04_01 to 2019_04_30.csv
⏳ Processing .csv/2019/2019_05_01 to 2019_05_31.csv
⏳ Processing .csv/2019/2019_06_01 to 2019_06_30.csv
⏳ Processing .csv/2019/2019_07_01 to 2019_07_31.csv
⏳ Processing .csv/2019/2019_08_01 to 2019_08_31.csv
⏳ Processing .csv/2019/2019_09_01 to 2019_09_30.csv
⏳ Processing .csv/2019/2019_10_01 to 2019_10_31.csv
⏳ Processing .csv/2019/2019_11_01 to 2019_11_30.csv
⏳ Processing .csv/2019/2019_12_01 to 2019_12_31.csv
⏳ Processing .csv/2020/01- 01-01-2020 TO 31-01-2020.csv
⏳ Processing .csv/2020/02- 01-02-2020 TO 29-02-2020.csv
⏳ Processing .csv/2020/03 - 01-03-2020 TO 31-03-2020.csv
⏳ Processing .csv/2020/04 - 01-04-2020 TO 30-04-2020.csv
⏳ Processing .csv/2020/05 - 01-05-2020 TO 31-05-2020.csv
⏳ Processing .csv/2020/06 - 2020_06_01 to 2020_06_30.csv
⏳ Processing .csv/2020/07 - 01-07-20

In [14]:
import boto3
import os

# ─── Config ────────────────────────────────────────────────────────────────
FILE_PATH   = 'cleaned_merged_raw.csv'
BUCKET_NAME = 'sagemakerstack-transactionsrawdatabucket6643d2da-q7nfqsc3jpnn'
S3_KEY      = 'merged/cleaned_merged_raw.csv'
PART_SIZE   = 100 * 1024 * 1024  # 100MB

# ─── Initialize ─────────────────────────────────────────────────────────────
s3 = boto3.client('s3')
file_size = os.path.getsize(FILE_PATH)
part_count = (file_size + PART_SIZE - 1) // PART_SIZE
print(f"📦 Uploading {FILE_PATH} in {part_count} parts...")

# ─── 1. Initiate multipart upload ───────────────────────────────────────────
mpu = s3.create_multipart_upload(Bucket=BUCKET_NAME, Key=S3_KEY)
upload_id = mpu['UploadId']
parts = []

# ─── 2. Upload each part ────────────────────────────────────────────────────
with open(FILE_PATH, 'rb') as f:
    for i in range(part_count):
        part_num = i + 1
        data = f.read(PART_SIZE)
        print(f"🔄 Uploading part {part_num} of {part_count}...")

        resp = s3.upload_part(
            Body=data,
            Bucket=BUCKET_NAME,
            Key=S3_KEY,
            UploadId=upload_id,
            PartNumber=part_num
        )
        parts.append({'PartNumber': part_num, 'ETag': resp['ETag']})

# ─── 3. Complete upload ─────────────────────────────────────────────────────
s3.complete_multipart_upload(
    Bucket=BUCKET_NAME,
    Key=S3_KEY,
    UploadId=upload_id,
    MultipartUpload={'Parts': parts}
)

print(f"✅ Upload complete: s3://{BUCKET_NAME}/{S3_KEY}")


📦 Uploading cleaned_merged_raw.csv in 67 parts...
🔄 Uploading part 1 of 67...
🔄 Uploading part 2 of 67...
🔄 Uploading part 3 of 67...
🔄 Uploading part 4 of 67...
🔄 Uploading part 5 of 67...
🔄 Uploading part 6 of 67...
🔄 Uploading part 7 of 67...
🔄 Uploading part 8 of 67...
🔄 Uploading part 9 of 67...
🔄 Uploading part 10 of 67...
🔄 Uploading part 11 of 67...
🔄 Uploading part 12 of 67...
🔄 Uploading part 13 of 67...
🔄 Uploading part 14 of 67...
🔄 Uploading part 15 of 67...
🔄 Uploading part 16 of 67...
🔄 Uploading part 17 of 67...
🔄 Uploading part 18 of 67...
🔄 Uploading part 19 of 67...
🔄 Uploading part 20 of 67...
🔄 Uploading part 21 of 67...
🔄 Uploading part 22 of 67...
🔄 Uploading part 23 of 67...
🔄 Uploading part 24 of 67...
🔄 Uploading part 25 of 67...
🔄 Uploading part 26 of 67...
🔄 Uploading part 27 of 67...
🔄 Uploading part 28 of 67...
🔄 Uploading part 29 of 67...
🔄 Uploading part 30 of 67...
🔄 Uploading part 31 of 67...
🔄 Uploading part 32 of 67...
🔄 Uploading part 33 of 67...
🔄 

In [16]:
import boto3
import pandas as pd
import io

BUCKET = 'sagemakerstack-transactionsrawdatabucket6643d2da-q7nfqsc3jpnn'
KEY    = 'merged/cleaned_merged_raw.csv'

s3 = boto3.client('s3')
obj = s3.get_object(Bucket=BUCKET, Key=KEY)

# ✅ Stream the first 1000 rows without reading the full file
df_sample = pd.read_csv(io.TextIOWrapper(obj['Body'], encoding='utf-8'), nrows=1000)

print("✅ Sample loaded:", df_sample.shape)
print("🧷 Columns:", list(df_sample.columns))



✅ Sample loaded: (1000, 73)
🧷 Columns: ['INDEX', 'Year', 'Customs Office Code', 'Customs Office Name', 'Regime', 'Registration Serial', 'Registration Number', 'Reference Number', 'Registration Date', 'Registration Time', 'Declarant CR', 'Declarant Name', 'Agent ID', 'Agent Name', 'Consignee CR', 'Consignee Name', 'Exporter CR', 'Exporter Name', 'Receipt Serial', 'Receipt Number', 'Receipt Date', 'Receipt Time', 'Cashier ID', 'Cashier Name', 'LOC Code', 'LOC Name', 'Terms of delivery', 'Office Exit', 'Office Exit Name', 'Item Number', 'Procedure', 'CP3', 'Pref', 'HSCode', 'Commercial Description', 'Country of Origin Code', 'Country of Origin', 'Country of Export Code', 'Country of Export', 'Country of Destination Code', 'Country of Destination', 'Invoice Currency', 'Invoice Amount', 'Local Currency', 'Local Amount', 'Customs Duty Rate', 'Customs Duty BHD', 'Excise Duty Rate', 'Excise Duty BHD', 'VAT Rate', 'VAT BHD', 'Total Duty BHD', 'HS-Rate', 'Fees', 'Currency', 'Gross Weight', 'Net 

In [17]:
# Check for repeated header rows (just in case)
header_rows = df_sample[df_sample.columns[0]] == df_sample.columns[0]
print("⚠️ Repeated headers found:" if header_rows.any() else "✅ No repeated headers found.")

✅ No repeated headers found.
